# 01 - Improved Dual-View Training

This notebook implements the first seven improvement steps on top of the original baseline:

1. target standardization,
2. improved training schedule and regression loss,
3. uniform sampling across the full sequence,
4. global train-split normalization instead of per-sequence max scaling,
5. stratified train/validation split,
6. explicit naive baseline comparisons,
7. joint use of `2CH` and `4CH` views.

All artifacts are saved under `improved model/results/` so this pipeline stays isolated from the original baseline outputs.


In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Model, layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import Sequence

warnings.filterwarnings('ignore')
plt.style.use('ggplot')


In [ ]:
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / 'data' / 'raw' / 'echo2022').exists():
        return cwd
    if cwd.name == 'improved model' and (cwd.parent / 'data' / 'raw' / 'echo2022').exists():
        return cwd.parent
    if (cwd.parent / 'data' / 'raw' / 'echo2022').exists():
        return cwd.parent
    raise FileNotFoundError('Could not resolve project root that contains data/raw/echo2022')


PROJECT_ROOT = resolve_project_root()
WORK_ROOT = PROJECT_ROOT / 'improved model'
DATA_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'echo2022'
TRAIN_CSV_PATH = DATA_ROOT / 'train_data.csv'
SAMPLE_SUB_PATH = DATA_ROOT / 'sample_submission.csv'
TRAIN_2CH_DIR = DATA_ROOT / 'train_data' / '2CH'
TRAIN_4CH_DIR = DATA_ROOT / 'train_data' / '4CH'
TEST_2CH_DIR = DATA_ROOT / 'test_data' / '2CH'
TEST_4CH_DIR = DATA_ROOT / 'test_data' / '4CH'

RESULTS_DIR = WORK_ROOT / 'results'
MODELS_DIR = RESULTS_DIR / 'models'
METRICS_DIR = RESULTS_DIR / 'metrics'
FIGURES_DIR = RESULTS_DIR / 'figures'
METADATA_DIR = RESULTS_DIR / 'metadata'

for path in [WORK_ROOT, RESULTS_DIR, MODELS_DIR, METRICS_DIR, FIGURES_DIR, METADATA_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('WORK_ROOT =', WORK_ROOT)
print('TRAIN_CSV_PATH exists =', TRAIN_CSV_PATH.exists())
print('TRAIN_2CH_DIR exists =', TRAIN_2CH_DIR.exists())
print('TRAIN_4CH_DIR exists =', TRAIN_4CH_DIR.exists())
print('TEST_2CH_DIR exists =', TEST_2CH_DIR.exists())
print('TEST_4CH_DIR exists =', TEST_4CH_DIR.exists())


In [ ]:
@dataclass(frozen=True)
class ImprovedConfig:
    img_size: tuple[int, int] = (160, 160)
    n_frames: int = 24
    batch_size: int = 4
    epochs: int = 50
    learning_rate: float = 3e-4
    dropout_rate: float = 0.30
    lstm_units: int = 64
    dense_units: int = 128
    val_size: float = 0.20
    lr_patience: int = 3
    early_stop_patience: int = 8
    stats_max_patients: int = 160
    stats_frames_per_view: int = 8
    stats_resize: tuple[int, int] = (96, 96)
    clip_percentiles: tuple[float, float] = (1.0, 99.0)
    handcrafted_frames: int = 12
    handcrafted_resize: tuple[int, int] = (96, 96)
    ridge_alpha: float = 1.0


CONFIG = ImprovedConfig()
CONFIG


In [ ]:
train_df = pd.read_csv(TRAIN_CSV_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

possible_id_cols = [col for col in train_df.columns if 'patient' in col.lower() or 'id' in col.lower()]
if not possible_id_cols:
    raise ValueError('Could not infer the patient id column from train_data.csv')

id_col = possible_id_cols[0]
train_df = train_df.copy()
train_df['patient_id'] = 'patient' + train_df[id_col].astype(str).str.extract(r'(\d+)', expand=False).str.zfill(3)
label_map = dict(zip(train_df['patient_id'], train_df['LV_ef'].astype(float)))


def collect_sequence_map(directory: Path) -> dict[str, Path]:
    return {path.stem.split('_')[0]: path for path in sorted(directory.glob('*.npy'))}


train_2ch_map = collect_sequence_map(TRAIN_2CH_DIR)
train_4ch_map = collect_sequence_map(TRAIN_4CH_DIR)
test_2ch_map = collect_sequence_map(TEST_2CH_DIR)
test_4ch_map = collect_sequence_map(TEST_4CH_DIR)

paired_train_ids = sorted(set(label_map) & set(train_2ch_map) & set(train_4ch_map))
paired_test_ids = sorted(set(test_2ch_map) & set(test_4ch_map))

train_records_df = pd.DataFrame(
    {
        'patient_id': paired_train_ids,
        'filepath_2ch': [str(train_2ch_map[pid]) for pid in paired_train_ids],
        'filepath_4ch': [str(train_4ch_map[pid]) for pid in paired_train_ids],
        'label': [float(label_map[pid]) for pid in paired_train_ids],
    }
)

test_records_df = sample_sub[['Patient_number']].rename(columns={'Patient_number': 'patient_id'}).copy()
test_records_df['filepath_2ch'] = test_records_df['patient_id'].map(lambda pid: str(test_2ch_map[pid]))
test_records_df['filepath_4ch'] = test_records_df['patient_id'].map(lambda pid: str(test_4ch_map[pid]))
if test_records_df[['filepath_2ch', 'filepath_4ch']].isna().any().any():
    missing = test_records_df[test_records_df[['filepath_2ch', 'filepath_4ch']].isna().any(axis=1)]['patient_id'].tolist()
    raise ValueError(f'Missing paired test views for: {missing[:5]}')

print('train_df shape =', train_df.shape)
print('paired train samples =', len(train_records_df))
print('paired test samples =', len(test_records_df))
print('label mean =', round(train_records_df['label'].mean(), 4))
print('label std =', round(train_records_df['label'].std(), 4))
train_records_df.head()


In [ ]:
def make_stratify_labels(labels: pd.Series, max_bins: int = 5) -> tuple[pd.Series | None, int]:
    for q in range(max_bins, 1, -1):
        try:
            binned = pd.qcut(labels, q=q, labels=False, duplicates='drop')
        except ValueError:
            continue
        counts = pd.Series(binned).value_counts(dropna=False)
        if not counts.empty and counts.min() >= 2:
            return binned.astype(int), int(counts.size)
    return None, 1


stratify_labels, n_bins = make_stratify_labels(train_records_df['label'])
train_split_df, val_split_df = train_test_split(
    train_records_df,
    test_size=CONFIG.val_size,
    random_state=SEED,
    stratify=stratify_labels,
)

train_split_df = train_split_df.sort_values('patient_id').reset_index(drop=True)
val_split_df = val_split_df.sort_values('patient_id').reset_index(drop=True)

train_split_df.to_csv(METADATA_DIR / 'train_split.csv', index=False)
val_split_df.to_csv(METADATA_DIR / 'val_split.csv', index=False)
test_records_df.to_csv(METADATA_DIR / 'test_records.csv', index=False)

split_summary = pd.DataFrame(
    {
        'split': ['train', 'val'],
        'n_samples': [len(train_split_df), len(val_split_df)],
        'label_mean': [train_split_df['label'].mean(), val_split_df['label'].mean()],
        'label_std': [train_split_df['label'].std(ddof=0), val_split_df['label'].std(ddof=0)],
        'label_min': [train_split_df['label'].min(), val_split_df['label'].min()],
        'label_max': [train_split_df['label'].max(), val_split_df['label'].max()],
    }
)
split_summary.to_csv(METRICS_DIR / 'split_summary.csv', index=False)

print('stratified bins used =', n_bins)
print(split_summary)


In [ ]:
def sample_frame_indices(n_available: int, n_frames: int) -> np.ndarray:
    if n_available <= 0:
        raise ValueError('Encountered an empty sequence')
    if n_available == 1:
        return np.zeros(n_frames, dtype=int)
    positions = np.linspace(0, n_available - 1, num=n_frames)
    return np.clip(np.round(positions).astype(int), 0, n_available - 1)


def load_raw_sequence(filepath: str | Path) -> np.ndarray:
    sequence = np.load(filepath).astype(np.float32)
    if sequence.ndim != 3:
        raise ValueError(f'Expected a 3D sequence, got shape {sequence.shape} for {filepath}')
    if sequence.shape[0] == 0:
        raise ValueError(f'Empty sequence found at {filepath}')
    return sequence


def resize_frames(frames: np.ndarray, img_size: tuple[int, int]) -> np.ndarray:
    height, width = img_size
    resized = np.zeros((len(frames), height, width), dtype=np.float32)
    for index, frame in enumerate(frames):
        resized[index] = cv2.resize(frame, (width, height), interpolation=cv2.INTER_LINEAR)
    return resized


def gather_normalization_stats(
    dataframe: pd.DataFrame,
    max_patients: int,
    frames_per_view: int,
    resize: tuple[int, int],
    clip_percentiles: tuple[float, float],
    seed: int,
) -> dict[str, float]:
    rng = np.random.default_rng(seed)
    sampled_df = dataframe.sample(n=min(len(dataframe), max_patients), random_state=seed)
    pixel_chunks: list[np.ndarray] = []

    for _, row in sampled_df.iterrows():
        for view_col in ['filepath_2ch', 'filepath_4ch']:
            sequence = load_raw_sequence(row[view_col])
            indices = sample_frame_indices(sequence.shape[0], min(frames_per_view, sequence.shape[0]))
            sampled_frames = resize_frames(sequence[indices], resize)
            flat = sampled_frames.reshape(-1)
            if flat.size > 4096:
                flat = rng.choice(flat, size=4096, replace=False)
            pixel_chunks.append(flat.astype(np.float32))

    pixels = np.concatenate(pixel_chunks, axis=0)
    clip_low, clip_high = np.percentile(pixels, clip_percentiles)
    clip_high = max(float(clip_high), float(clip_low) + 1e-6)
    scaled_pixels = (np.clip(pixels, clip_low, clip_high) - clip_low) / (clip_high - clip_low)
    mean = float(scaled_pixels.mean())
    std = float(scaled_pixels.std() + 1e-6)

    return {
        'clip_low': float(clip_low),
        'clip_high': float(clip_high),
        'scaled_mean': mean,
        'scaled_std': std,
        'sampled_patients': int(len(sampled_df)),
        'frames_per_view': int(frames_per_view),
        'stats_height': int(resize[0]),
        'stats_width': int(resize[1]),
    }


norm_stats = gather_normalization_stats(
    dataframe=train_split_df,
    max_patients=CONFIG.stats_max_patients,
    frames_per_view=CONFIG.stats_frames_per_view,
    resize=CONFIG.stats_resize,
    clip_percentiles=CONFIG.clip_percentiles,
    seed=SEED,
)

print(norm_stats)


In [ ]:
def preprocess_sequence(
    filepath: str | Path,
    n_frames: int,
    img_size: tuple[int, int],
    norm_stats: dict[str, float],
) -> np.ndarray:
    sequence = load_raw_sequence(filepath)
    indices = sample_frame_indices(sequence.shape[0], n_frames)
    sampled = sequence[indices]
    resized = resize_frames(sampled, img_size)

    low = norm_stats['clip_low']
    high = norm_stats['clip_high']
    scaled = (np.clip(resized, low, high) - low) / (high - low)
    normalized = (scaled - norm_stats['scaled_mean']) / norm_stats['scaled_std']
    return normalized[..., np.newaxis].astype(np.float32)


class DualViewSequence(Sequence):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        config: ImprovedConfig,
        norm_stats: dict[str, float],
        target_mean: float | None = None,
        target_std: float | None = None,
        shuffle: bool = True,
        include_targets: bool = True,
    ) -> None:
        self.dataframe = dataframe.reset_index(drop=True)
        self.config = config
        self.norm_stats = norm_stats
        self.target_mean = target_mean
        self.target_std = target_std
        self.shuffle = shuffle
        self.include_targets = include_targets
        self.indices = np.arange(len(self.dataframe))
        self.on_epoch_end()

    def __len__(self) -> int:
        return math.ceil(len(self.dataframe) / self.config.batch_size)

    def __getitem__(self, batch_index: int):
        start = batch_index * self.config.batch_size
        stop = start + self.config.batch_size
        batch_indices = self.indices[start:stop]
        batch_df = self.dataframe.iloc[batch_indices]

        height, width = self.config.img_size
        batch_2ch = np.zeros((len(batch_df), self.config.n_frames, height, width, 1), dtype=np.float32)
        batch_4ch = np.zeros_like(batch_2ch)

        for row_idx, (_, row) in enumerate(batch_df.iterrows()):
            batch_2ch[row_idx] = preprocess_sequence(row['filepath_2ch'], self.config.n_frames, self.config.img_size, self.norm_stats)
            batch_4ch[row_idx] = preprocess_sequence(row['filepath_4ch'], self.config.n_frames, self.config.img_size, self.norm_stats)

        inputs = {'input_2ch': batch_2ch, 'input_4ch': batch_4ch}
        if not self.include_targets:
            return inputs

        if self.target_mean is None or self.target_std is None:
            raise ValueError('target_mean and target_std are required when include_targets=True')

        targets = batch_df['label'].to_numpy(dtype=np.float32)
        scaled_targets = ((targets - self.target_mean) / self.target_std).reshape(-1, 1)
        return inputs, scaled_targets.astype(np.float32)

    def on_epoch_end(self) -> None:
        if self.shuffle:
            np.random.shuffle(self.indices)


In [ ]:
def compute_metrics(targets: np.ndarray, predictions: np.ndarray) -> dict[str, float]:
    return {
        'rmse': float(np.sqrt(mean_squared_error(targets, predictions))),
        'mae': float(mean_absolute_error(targets, predictions)),
        'r2': float(r2_score(targets, predictions)),
    }


def inverse_scale_targets(values: np.ndarray, mean: float, std: float) -> np.ndarray:
    return values * std + mean


def summarize_predictions(targets: np.ndarray, predictions: np.ndarray) -> pd.DataFrame:
    errors = predictions - targets
    rows = [
        ('Validation RMSE', float(np.sqrt(mean_squared_error(targets, predictions)))),
        ('Validation MAE', float(mean_absolute_error(targets, predictions))),
        ('Validation R2', float(r2_score(targets, predictions))),
        ('Prediction mean', float(predictions.mean())),
        ('Prediction std', float(predictions.std())),
        ('Target mean', float(targets.mean())),
        ('Target std', float(targets.std())),
        ('Prediction min', float(predictions.min())),
        ('Prediction max', float(predictions.max())),
        ('Target min', float(targets.min())),
        ('Target max', float(targets.max())),
        ('Median abs error', float(np.median(np.abs(errors)))),
        ('Best-case abs error', float(np.abs(errors).min())),
        ('Worst-case abs error', float(np.abs(errors).max())),
    ]
    return pd.DataFrame(rows, columns=['metric', 'value'])


def compute_handcrafted_view_features(filepath: str | Path, frames: int, resize: tuple[int, int], norm_stats: dict[str, float]) -> np.ndarray:
    sequence = preprocess_sequence(filepath, frames, resize, norm_stats).squeeze(-1)
    frame_means = sequence.mean(axis=(1, 2))
    frame_stds = sequence.std(axis=(1, 2))
    temporal_diff = np.diff(sequence, axis=0)

    features = [
        float(sequence.mean()),
        float(sequence.std()),
        float(sequence.min()),
        float(sequence.max()),
        float(np.percentile(sequence, 10)),
        float(np.percentile(sequence, 90)),
        float(frame_means.mean()),
        float(frame_means.std()),
        float(frame_stds.mean()),
        float(frame_stds.std()),
        float(np.abs(temporal_diff).mean()) if len(temporal_diff) else 0.0,
        float(np.abs(temporal_diff).std()) if len(temporal_diff) else 0.0,
    ]
    return np.array(features, dtype=np.float32)


def build_handcrafted_feature_matrix(dataframe: pd.DataFrame, config: ImprovedConfig, norm_stats: dict[str, float]) -> np.ndarray:
    rows = []
    for _, row in dataframe.iterrows():
        features_2ch = compute_handcrafted_view_features(row['filepath_2ch'], config.handcrafted_frames, config.handcrafted_resize, norm_stats)
        features_4ch = compute_handcrafted_view_features(row['filepath_4ch'], config.handcrafted_frames, config.handcrafted_resize, norm_stats)
        combined = np.concatenate([features_2ch, features_4ch, np.abs(features_2ch - features_4ch)], axis=0)
        rows.append(combined)
    return np.vstack(rows)


In [ ]:
target_mean = float(train_split_df['label'].mean())
target_std = float(train_split_df['label'].std(ddof=0) + 1e-6)

metadata = {
    'seed': SEED,
    'config': {
        key: list(value) if isinstance(value, tuple) else value
        for key, value in asdict(CONFIG).items()
    },
    'target_stats': {
        'mean': target_mean,
        'std': target_std,
    },
    'normalization_stats': norm_stats,
}
(METADATA_DIR / 'preprocessing_metadata.json').write_text(json.dumps(metadata, indent=2))

train_features = build_handcrafted_feature_matrix(train_split_df, CONFIG, norm_stats)
val_features = build_handcrafted_feature_matrix(val_split_df, CONFIG, norm_stats)

ridge_model = make_pipeline(StandardScaler(), Ridge(alpha=CONFIG.ridge_alpha))
ridge_model.fit(train_features, train_split_df['label'].to_numpy(dtype=np.float32))
ridge_val_predictions = ridge_model.predict(val_features).astype(np.float32)

mean_baseline_predictions = np.full(len(val_split_df), target_mean, dtype=np.float32)
val_targets = val_split_df['label'].to_numpy(dtype=np.float32)

baseline_comparison = pd.DataFrame([
    {'model': 'train_mean_baseline', **compute_metrics(val_targets, mean_baseline_predictions)},
    {'model': 'ridge_handcrafted_baseline', **compute_metrics(val_targets, ridge_val_predictions)},
])
baseline_comparison.to_csv(METRICS_DIR / 'baseline_comparison.csv', index=False)

baseline_comparison


In [ ]:
def build_frame_encoder(frame_shape: tuple[int, int, int]) -> Model:
    inputs = layers.Input(shape=frame_shape)
    x = layers.Conv2D(24, (3, 3), padding='same', activation='swish')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(48, (3, 3), padding='same', activation='swish')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(96, (3, 3), padding='same', activation='swish')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation='swish')(x)
    return Model(inputs, x, name='shared_frame_encoder')


def build_improved_model(config: ImprovedConfig) -> Model:
    height, width = config.img_size
    frame_encoder = build_frame_encoder((height, width, 1))

    input_2ch = layers.Input(shape=(config.n_frames, height, width, 1), name='input_2ch')
    input_4ch = layers.Input(shape=(config.n_frames, height, width, 1), name='input_4ch')

    seq_2ch = layers.TimeDistributed(frame_encoder, name='td_2ch')(input_2ch)
    seq_4ch = layers.TimeDistributed(frame_encoder, name='td_4ch')(input_4ch)

    seq_2ch = layers.Bidirectional(layers.LSTM(config.lstm_units, return_sequences=True), name='bilstm_2ch')(seq_2ch)
    seq_4ch = layers.Bidirectional(layers.LSTM(config.lstm_units, return_sequences=True), name='bilstm_4ch')(seq_4ch)

    pooled_2ch = layers.GlobalAveragePooling1D(name='gap_2ch')(seq_2ch)
    pooled_4ch = layers.GlobalAveragePooling1D(name='gap_4ch')(seq_4ch)
    difference = layers.Lambda(lambda tensors: tf.math.abs(tensors[0] - tensors[1]), name='view_gap')([pooled_2ch, pooled_4ch])

    x = layers.Concatenate(name='fusion')([pooled_2ch, pooled_4ch, difference])
    x = layers.Dense(config.dense_units, activation='swish')(x)
    x = layers.Dropout(config.dropout_rate)(x)
    x = layers.Dense(64, activation='swish')(x)
    x = layers.Dropout(config.dropout_rate * 0.5)(x)
    outputs = layers.Dense(1, activation='linear', name='scaled_ef')(x)

    model = Model(inputs=[input_2ch, input_4ch], outputs=outputs, name='improved_dual_view_regressor')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=config.learning_rate),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse'), tf.keras.metrics.MeanAbsoluteError(name='mae')],
    )
    return model


model = build_improved_model(CONFIG)
model.summary()


In [ ]:
train_gen = DualViewSequence(
    dataframe=train_split_df,
    config=CONFIG,
    norm_stats=norm_stats,
    target_mean=target_mean,
    target_std=target_std,
    shuffle=True,
    include_targets=True,
)

val_gen = DualViewSequence(
    dataframe=val_split_df,
    config=CONFIG,
    norm_stats=norm_stats,
    target_mean=target_mean,
    target_std=target_std,
    shuffle=False,
    include_targets=True,
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=CONFIG.early_stop_patience, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=CONFIG.lr_patience, min_lr=1e-6, verbose=1),
    ModelCheckpoint(filepath=str(MODELS_DIR / 'improved_dual_view_best.keras'), monitor='val_loss', save_best_only=True),
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=CONFIG.epochs,
    callbacks=callbacks,
    verbose=1,
)

history_df = pd.DataFrame(history.history)
history_df.to_csv(METRICS_DIR / 'improved_history.csv', index=False)
history_df.tail()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df['loss'], label='train_loss')
axes[0].plot(history_df['val_loss'], label='val_loss')
axes[0].set_title('Training Loss')
axes[0].legend()

axes[1].plot(history_df['rmse'], label='train_rmse')
axes[1].plot(history_df['val_rmse'], label='val_rmse')
axes[1].set_title('Scaled RMSE')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'improved_training_curves.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
best_model = tf.keras.models.load_model(MODELS_DIR / 'improved_dual_view_best.keras')

val_pred_gen = DualViewSequence(
    dataframe=val_split_df,
    config=CONFIG,
    norm_stats=norm_stats,
    target_mean=target_mean,
    target_std=target_std,
    shuffle=False,
    include_targets=False,
)

val_scaled_predictions = best_model.predict(val_pred_gen, verbose=1).reshape(-1)
val_predictions = inverse_scale_targets(val_scaled_predictions, target_mean, target_std)
val_targets = val_split_df['label'].to_numpy(dtype=np.float32)

validation_predictions_df = val_split_df.copy()
validation_predictions_df['prediction'] = val_predictions
validation_predictions_df['error'] = validation_predictions_df['prediction'] - validation_predictions_df['label']
validation_predictions_df['abs_error'] = validation_predictions_df['error'].abs()
validation_predictions_df.to_csv(METRICS_DIR / 'validation_predictions.csv', index=False)

prediction_summary_df = summarize_predictions(val_targets, val_predictions)
prediction_summary_df.to_csv(METRICS_DIR / 'prediction_summary.csv', index=False)

model_metrics = compute_metrics(val_targets, val_predictions)
comparison_with_model = pd.concat(
    [baseline_comparison, pd.DataFrame([{'model': 'improved_dual_view_model', **model_metrics}])],
    ignore_index=True,
)
comparison_with_model.to_csv(METRICS_DIR / 'model_comparison.csv', index=False)

validation_predictions_df.head()


In [ ]:
def summarize_extreme_cases(results_df: pd.DataFrame, n_cases: int = 10) -> tuple[pd.DataFrame, pd.DataFrame]:
    best_cases = results_df.sort_values('abs_error').head(n_cases).reset_index(drop=True)
    worst_cases = results_df.sort_values('abs_error', ascending=False).head(n_cases).reset_index(drop=True)
    return best_cases, worst_cases


def assign_ef_group(value: float) -> str:
    if value < 40:
        return 'Low EF (<40)'
    if value < 55:
        return 'Mid EF (40-55)'
    return 'High EF (>=55)'


validation_predictions_df['ef_group'] = validation_predictions_df['label'].map(assign_ef_group)
ef_group_summary_df = (
    validation_predictions_df.groupby('ef_group', observed=False)
    .agg(
        n_samples=('patient_id', 'count'),
        mean_target_ef=('label', 'mean'),
        mean_prediction=('prediction', 'mean'),
        mean_error=('error', 'mean'),
        mean_abs_error=('abs_error', 'mean'),
        median_abs_error=('abs_error', 'median'),
        max_abs_error=('abs_error', 'max'),
    )
    .reset_index()
)
ef_group_summary_df.to_csv(METRICS_DIR / 'ef_group_error_summary.csv', index=False)

best_cases_df, worst_cases_df = summarize_extreme_cases(validation_predictions_df, n_cases=10)
best_cases_df.to_csv(METRICS_DIR / 'top10_best_validation_cases.csv', index=False)
worst_cases_df.to_csv(METRICS_DIR / 'top10_worst_validation_cases.csv', index=False)

ef_group_summary_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(validation_predictions_df['label'], validation_predictions_df['prediction'], alpha=0.75)
lims = [
    min(validation_predictions_df['label'].min(), validation_predictions_df['prediction'].min()),
    max(validation_predictions_df['label'].max(), validation_predictions_df['prediction'].max()),
]
axes[0].plot(lims, lims, '--', color='black', linewidth=1)
axes[0].set_title('Validation: Ground Truth vs Prediction')
axes[0].set_xlabel('Ground Truth EF')
axes[0].set_ylabel('Predicted EF')

axes[1].hist(validation_predictions_df['error'], bins=20, color='#4C72B0', alpha=0.85)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Validation Error Distribution')
axes[1].set_xlabel('Prediction - Target')

axes[2].hist(validation_predictions_df['abs_error'], bins=20, color='#55A868', alpha=0.85)
axes[2].set_title('Absolute Error Distribution')
axes[2].set_xlabel('|Prediction - Target|')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'validation_overview.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
test_pred_gen = DualViewSequence(
    dataframe=test_records_df,
    config=CONFIG,
    norm_stats=norm_stats,
    target_mean=target_mean,
    target_std=target_std,
    shuffle=False,
    include_targets=False,
)

test_scaled_predictions = best_model.predict(test_pred_gen, verbose=1).reshape(-1)
test_predictions = inverse_scale_targets(test_scaled_predictions, target_mean, target_std)

submission_df = sample_sub.copy()
submission_target_col = [col for col in submission_df.columns if col != submission_df.columns[0]][0]
submission_df[submission_target_col] = test_predictions
submission_df.to_csv(RESULTS_DIR / 'improved_submission.csv', index=False)

prediction_summary_df


## Saved Outputs

After this notebook finishes successfully, the improved pipeline writes:

- `improved model/results/models/improved_dual_view_best.keras`
- `improved model/results/metrics/improved_history.csv`
- `improved model/results/metrics/model_comparison.csv`
- `improved model/results/metrics/validation_predictions.csv`
- `improved model/results/metrics/prediction_summary.csv`
- `improved model/results/metrics/ef_group_error_summary.csv`
- `improved model/results/improved_submission.csv`
- `improved model/results/figures/improved_training_curves.png`
- `improved model/results/figures/validation_overview.png`
